# Loop1, stage 00: read the wells, label them, concatenate

Plate 1 of experiment LPHO013, twelve experiments in columns 1 to 12.

## Inputs

| file | produced by | holds |
|---|---|---|
| `metadata.csv` | this folder | culture conditions, one row per experiment |
| `$LABCOMPASS_FCS_ROOT/LPHO013/plate1/filtered` | acquisition, outside the repository | one unmixed `.fcs` file per well |
| `live.npz` | this folder | gates for the live-cell count per well |
| `counts.npz` | this folder | gates for the counting-bead count per well |

## Outputs

| file | holds |
|---|---|
| `Loop1.h5ad` | every cell of every well, labelled, with the antibody channels only |
| `Loop1_500k.h5ad` | about 500,000 of those cells, spread evenly over the experiments |

Run the cells from top to bottom.  Nothing below reads a file that a later cell
writes, so `Restart Kernel and Run All Cells` reproduces both outputs.

## Setup

Every path and every number that this notebook uses is set here.

In [ ]:
import gc
import os

import pytometry as pm

from labcompass import (
    add_live_dead_obs,
    apply_saved_gates,
    attach_condition_metadata,
    attach_cytometer_metadata,
    concat_wells,
    downsample,
    drop_channels,
    find_fcs_files,
    load_metadata_table,
    read_well,
    rename_channels_to_antibody,
    target_size_for,
)

# ---------------------------------------------------------------- configuration
# Where the unmixed .fcs files live.  They are far too large to commit, so the
# repository only records the path.  Override it without editing this notebook
# by setting the environment variable LABCOMPASS_FCS_ROOT.
FCS_ROOT = os.environ.get("LABCOMPASS_FCS_ROOT", '/rds/project/rds-SDzz0CATGms/unsorted/for_Juan_Licyel')
DATA_PATH = os.path.join(FCS_ROOT, 'LPHO013/plate1/filtered')
METADATA_CSV = 'metadata.csv'

# One experiment per plate column.  Entry i of EXPERIMENTS is the
# experiment in column i + 1, so the order of this list is part of the data.
EXPERIMENTS = [
    '216', '217', '218', '219', '220', '221',
    '222', '223', '224', '225', '226', '227',
]

# Gates used per well to count live cells and counting beads.  Both are
# applied to the raw well, before the scatter channels move into .obs.
LIVE_GATES = "live.npz"
COUNT_GATES = "counts.npz"

OUT_ALL = "Loop1.h5ad"
OUT_SUBSET = "Loop1_500k.h5ad"

# Cap the downsampled file at this many cells in total, spread evenly over the
# experiments.  It exists so the gating notebook has something small to plot.
DESIRED_CELL_NUMBER = 500000
DOWNSAMPLE_SEED = 0

## The wells to read

Every unmixed `.fcs` file below `DATA_PATH` is read.

The list is sorted, so the order does not depend on the filesystem.

In [ ]:
files = find_fcs_files(DATA_PATH, subdir=None)
print(f"{len(files)} wells to read")
if not files:
    raise FileNotFoundError(f"no .fcs files found under {DATA_PATH}")

## The condition table

One row per experiment, holding the culture conditions.  The index is read as
text, so an experiment number matches whether it is written as `239` or `"239"`.

In [ ]:
metadata = load_metadata_table(METADATA_CSV)
print(f"{len(metadata)} experiments, {len(metadata.columns)} condition columns")
metadata.head()

## Read each well and label it

The plate column gives the experiment and the plate row gives the replicate.
Well `C05` is therefore column 5, which is `EXPERIMENTS[4]`, and row `C`, which
is replicate 1 counting from row `B`.

Two counts are recorded per well before any channel is dropped.  `cell_counts`
is the number of live cells, from the gates in `live.npz`.  `count_beads` is the
number of counting beads, from the gates in `counts.npz`.  Their ratio converts
an event count into an absolute cell number.

`pytometry.pp.split_signal` moves the scatter and time channels from `.X` into
`.obs` and keeps only the area measurements in `.X`.

In [ ]:
adatas = []
skipped = []

for path in files:
    adata = read_well(path)
    if adata.n_obs == 0:
        continue

    well_id = adata.uns["meta"]["wellid"]
    column = int(well_id[1:])          # "C05" -> 5
    replicate = ord(well_id[0]) - ord("B")   # row B is replicate 0, C is 1, and so on

    if not 1 <= column <= len(EXPERIMENTS):
        skipped.append((path, f"column {column} is outside EXPERIMENTS"))
        continue

    experiment_number = EXPERIMENTS[column - 1]
    if experiment_number not in metadata.index:
        skipped.append((path, f"experiment {experiment_number} not in {METADATA_CSV}"))
        continue

    # Counted on the raw well, while the scatter channels are still in .X.
    live_cells, _ = apply_saved_gates(adata, LIVE_GATES)
    beads, _ = apply_saved_gates(adata, COUNT_GATES)
    adata.obs["cell_counts"] = live_cells.n_obs
    adata.obs["count_beads"] = beads.n_obs

    adata.obs["experiment_number"] = experiment_number
    adata.obs["replicate"] = replicate
    attach_cytometer_metadata(adata)
    attach_condition_metadata(adata, metadata, experiment_number)

    pm.pp.split_signal(adata, var_key="channel", option="area", data_type="facs")
    adatas.append(adata)

del adata
gc.collect()

print(f"{len(adatas)} wells kept, {len(skipped)} skipped")
for path, reason in skipped:
    print("  skipped", path, "|", reason)

## Concatenate, move the live/dead channel, drop the channels we do not analyse

`concat_wells` keeps only the channels present in every well and makes the cell
names unique across wells.  The live/dead channel is copied into `.obs["7-AAD"]`
before it is dropped, because the live gate is drawn against it.

In [ ]:
adata_all = concat_wells(adatas)
del adatas
gc.collect()
adata_all

In [ ]:
add_live_dead_obs(adata_all)
adata_all = drop_channels(adata_all)
fluorochromes = rename_channels_to_antibody(adata_all)

print("antibodies:  ", list(adata_all.var_names))
print("fluorochromes:", fluorochromes)
adata_all

## Write the full object

This is the input of stage 02.

In [ ]:
adata_all.write(OUT_ALL)
print(f"wrote {OUT_ALL}: {adata_all.n_obs:,} cells x {adata_all.n_vars} antibodies")

## Write the downsampled object

Each experiment is capped at the same number of cells.  An experiment with fewer
cells than the cap is kept whole, so the total comes out below the target.

In [ ]:
target_size = target_size_for(adata_all, DESIRED_CELL_NUMBER, obs="experiment_number")
print(f"cap of {target_size:,} cells per experiment")

adata_small = downsample(
    adata_all, obs="experiment_number", target_size=target_size, seed=DOWNSAMPLE_SEED
)
adata_small.write(OUT_SUBSET)
print(f"wrote {OUT_SUBSET}: {adata_small.n_obs:,} cells")